<a href="https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("Connected, ready to query.")

Connected, ready to query.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm using a Decision Tree Classifier for this.

Why: my baseline (ML-07) was a simple one-condition rule comparing CTR to
an expected value per position bucket. A decision tree is the natural next
step up from that, it's still fully readable (I can print it as an
if/else, same as I did in Notebook 2), but it can combine more than one
signal at once instead of relying on one hand-picked threshold.

I checked GA4 field availability first, and found only about 3.7% of rows
have both GSC and GA4 data present. Since my baseline only used GSC data,
I'm sticking to GSC-only features here too, so the model and the baseline
are compared on the exact same slice of data, not a smaller GA4-only
subset that would make the comparison unfair.

Features: gsc_impressions, gsc_avg_position (both available whenever
gsc_data_available is true, same condition my baseline used).

Label: is_low_ctr, same definition as my baseline, whether a page's
actual CTR falls below what's expected for its position bucket.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

check = con.sql(f"""
    SELECT gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 50
    LIMIT 5
""").df()
print(check)
print("\nAny missing values?")
print(check.isna().sum())

   gsc_impressions  gsc_clicks  gsc_avg_position
0              125           1          4.928000
1              239           1          7.347280
2              191           0          7.832461
3               55           0          3.272727
4               77           0          5.636364

Any missing values?
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'm using a random train/test split (80/20), not a strict client-grouped
split, for one honest reason: my label (is_low_ctr) and features
(gsc_impressions, gsc_avg_position) are both daily, page-level snapshots
within the same month, not something that depends on a client's history
length or identity in a way that would leak across the split. A random
row-level split is appropriate here because each row already stands on
its own.

I did consider a client-grouped split (like the one used in the official
FlyRank pipeline from Notebook 1), but that matters more when the label
depends on trends over time for a specific client. Since my label here is
based on a single day's CTR-vs-position comparison, a plain random split
is enough to avoid leakage, and it's simpler and honest for this specific
question.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

model_data = con.sql(f"""
    SELECT gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 50
""").df()

# Rebuild the same label logic as the baseline (ML-07)
model_data["actual_ctr"] = model_data["gsc_clicks"] / model_data["gsc_impressions"]

def bucket(pos):
    if pos <= 3: return "top_3"
    elif pos <= 10: return "page_1"
    elif pos <= 20: return "page_2"
    else: return "deep"

model_data["position_bucket"] = model_data["gsc_avg_position"].apply(bucket)
expected_ctr_map = model_data.groupby("position_bucket")["actual_ctr"].mean().to_dict()
model_data["expected_ctr"] = model_data["position_bucket"].map(expected_ctr_map)
model_data["is_low_ctr"] = (model_data["actual_ctr"] < model_data["expected_ctr"]).astype(int)

X = model_data[["gsc_impressions", "gsc_avg_position"]]
y = model_data["is_low_ctr"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Label balance in train: {y_train.mean():.3f}")
print(f"Label balance in test: {y_test.mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train size: 829953, Test size: 207489
Label balance in train: 0.729
Label balance in test: 0.729


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Now training a Decision Tree on the same train/test split, then comparing
it against my baseline rule (ML-07) on the exact same test set and metric.

My baseline rule (from ML-07) was: flag a page as "review_ctr_fix" if its
actual CTR is below the expected CTR for its position bucket, which is
actually the exact same logic I used to define the is_low_ctr label here.
So to make this a fair comparison, I'm treating the baseline as: "always
predict is_low_ctr = 1 whenever actual_ctr < expected_ctr" (this is
circular by definition, so instead I'm comparing the tree against a
simpler, more honest baseline: predicting the majority class every time,
which is the standard "can a real model beat doing nothing smart" check).

Metric: accuracy, since the label is close to balanced (~73%/27% split,
not extremely skewed), and it directly answers "how often is this
right," matching the same plain-language framing I used in the baseline.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Baseline: majority-class dummy classifier (honest "no smarts" baseline)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# Model: Decision Tree, WITHOUT forced class balancing, for a fair accuracy comparison
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)

comparison = pd.DataFrame({
    "method": ["Baseline (majority class)", "Decision Tree (max_depth=4)"],
    "accuracy": [accuracy_score(y_test, baseline_pred), accuracy_score(y_test, tree_pred)],
    "precision (class=1)": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, tree_pred, zero_division=0)
    ],
    "recall (class=1)": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, tree_pred, zero_division=0)
    ]
})
print(comparison)

                        method  accuracy  precision (class=1)  \
0    Baseline (majority class)  0.728762             0.728762   
1  Decision Tree (max_depth=4)  0.728762             0.728762   

   recall (class=1)  
0               1.0  
1               1.0  


In [8]:
import numpy as np

# Check: is the tree just predicting one class for everyone?
print("Tree's predicted class distribution:")
print(pd.Series(tree_pred).value_counts())

print("\nBaseline's predicted class distribution:")
print(pd.Series(baseline_pred).value_counts())

# Look at feature importance to see if the tree is learning anything real
print("\nFeature importances:")
for feat, imp in zip(X.columns, tree.feature_importances_):
    print(f"{feat}: {imp:.4f}")

# Print the actual tree logic
from sklearn.tree import export_text
print("\nTree structure:")
print(export_text(tree, feature_names=list(X.columns)))

Tree's predicted class distribution:
1    207489
Name: count, dtype: int64

Baseline's predicted class distribution:
1    207489
Name: count, dtype: int64

Feature importances:
gsc_impressions: 0.7186
gsc_avg_position: 0.2814

Tree structure:
|--- gsc_impressions <= 123.50
|   |--- gsc_avg_position <= 21.20
|   |   |--- gsc_impressions <= 87.50
|   |   |   |--- gsc_avg_position <= 7.71
|   |   |   |   |--- class: 1
|   |   |   |--- gsc_avg_position >  7.71
|   |   |   |   |--- class: 1
|   |   |--- gsc_impressions >  87.50
|   |   |   |--- gsc_avg_position <= 6.98
|   |   |   |   |--- class: 1
|   |   |   |--- gsc_avg_position >  6.98
|   |   |   |   |--- class: 1
|   |--- gsc_avg_position >  21.20
|   |   |--- gsc_avg_position <= 33.17
|   |   |   |--- gsc_impressions <= 75.50
|   |   |   |   |--- class: 1
|   |   |   |--- gsc_impressions >  75.50
|   |   |   |   |--- class: 1
|   |   |--- gsc_avg_position >  33.17
|   |   |   |--- gsc_avg_position <= 43.42
|   |   |   |   |--- class:

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Errors and interpretation: my Decision Tree completely failed to beat the
baseline, and looking at the tree structure explains why. Every single
leaf in the tree predicts class 1 ("low CTR") no matter which branch a
page falls into. The tree tried plenty of splits (using both features,
with impressions weighted more heavily at 0.72 importance vs 0.28 for
position), but never found a split where the "not low CTR" class became
the majority in a leaf.

Why this happened: my label (is_low_ctr) is defined as being below the
*average* CTR for a page's own position bucket. That means, roughly
speaking, close to half of any bucket will always fall below its own
bucket's average, no matter what gsc_impressions or gsc_avg_position value
they have — position itself is already baked into how the label was
built, so using position again as a model feature doesn't add new signal.

The honest conclusion: gsc_impressions and gsc_avg_position alone aren't
enough to predict this particular label. To do better, I'd need features
that vary independently of position and impressions, like page-level
content signals (word count, content type) or historical patterns for the
same page. This isn't a failure of the modeling process, it's a real
result: it tells me my current feature set can't beat a naive guess for
this specific label, and that's useful information before spending more
time on this exact framing.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Tree always predicts class:", pd.Series(tree_pred).unique())
print("This matches the tree structure - every leaf ends in 'class: 1'.")
print("\nHonest conclusion: with only gsc_impressions and gsc_avg_position,")
print("neither the tree nor the baseline can beat simply guessing the majority class.")

Tree always predicts class: [1]
This matches the tree structure - every leaf ends in 'class: 1'.

Honest conclusion: with only gsc_impressions and gsc_avg_position,
neither the tree nor the baseline can beat simply guessing the majority class.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.